# 3: ETL / ELT Processing Pipeline
**Pipeline Stage:** Transform raw data into ML-ready features.

This notebook demonstrates:
- **Extract** — pull from the SQLite raw store
- **Transform** — cleaning, feature engineering, encoding, scaling, SMOTE for class imbalance
- **Load** — write processed data back to SQLite and Parquet
- Simulated **Airflow DAG** structure (using plain Python task functions)
- Simulated **Spark-style** partitioned processing using pandas chunking

In [1]:
!pip install pandas scikit-learn imbalanced-learn pyarrow sqlalchemy

In [2]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from imblearn.over_sampling import SMOTE

BASE_DIR   = Path('..').resolve()
OUTPUT_DIR = BASE_DIR / 'outputs'
DB_PATH    = OUTPUT_DIR / 'pipeline.db'
LAKE_DIR   = OUTPUT_DIR / 'data_lake'

print("Dependencies loaded.")

Dependencies loaded.


## 3.1  EXTRACT — Load from SQL Store

In [3]:
def task_extract() -> pd.DataFrame:
    """Airflow Task: extract raw data from SQL."""
    conn = sqlite3.connect(DB_PATH)
    # Use only CSV batch source (ground truth) for ML pipeline
    df = pd.read_sql("SELECT * FROM sessions WHERE ingestion_source='csv_batch'", conn)
    conn.close()
    print(f"[EXTRACT] {len(df):,} rows loaded")
    return df

df_raw = task_extract()
df_raw.info()

[EXTRACT] 12,205 rows loaded
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12205 entries, 0 to 12204
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12205 non-null  int64  
 1   Administrative_Duration  12205 non-null  float64
 2   Informational            12205 non-null  int64  
 3   Informational_Duration   12205 non-null  float64
 4   ProductRelated           12205 non-null  int64  
 5   ProductRelated_Duration  12205 non-null  float64
 6   BounceRates              12205 non-null  float64
 7   ExitRates                12205 non-null  float64
 8   PageValues               12205 non-null  float64
 9   SpecialDay               12205 non-null  float64
 10  Month                    12205 non-null  object 
 11  OperatingSystems         12205 non-null  int64  
 12  Browser                  12205 non-null  int64  
 13  Region                   12205 non-null  int64 

## 3.2  TRANSFORM — Step by Step

In [4]:
def task_clean(df: pd.DataFrame) -> pd.DataFrame:
    """Remove metadata columns, handle nulls, fix types."""
    drop_cols = ['id','row_id','ingestion_source','ingestion_timestamp','checksum']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # Fill any remaining nulls
    numeric_cols = df.select_dtypes(include='number').columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

    df['Revenue'] = df['Revenue'].astype(int)
    df['Weekend'] = df['Weekend'].astype(int)

    print(f"[CLEAN] Shape: {df.shape}  |  Nulls: {df.isnull().sum().sum()}")
    return df

df_cleaned = task_clean(df_raw)
df_cleaned.head(2)

[CLEAN] Shape: (12205, 18)  |  Nulls: 0


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,0,0
1,0,0.0,0,0.0,2,64.0,0.0,0.1,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,0,0


In [5]:
def task_feature_engineer(df: pd.DataFrame) -> pd.DataFrame:
    """Derive new informative features from raw columns."""
    df = df.copy()

    # Total pages visited
    df['TotalPages'] = df['Administrative'] + df['Informational'] + df['ProductRelated']

    # Total time on site (seconds)
    df['TotalDuration'] = (df['Administrative_Duration'] +
                           df['Informational_Duration'] +
                           df['ProductRelated_Duration'])

    # Average time per page (avoid division by zero)
    df['AvgDurationPerPage'] = df['TotalDuration'] / (df['TotalPages'] + 1)

    # Product focus ratio
    df['ProductFocusRatio'] = df['ProductRelated'] / (df['TotalPages'] + 1)

    # Engagement score (inverse of bounce + exit rates)
    df['EngagementScore'] = 1 - (df['BounceRates'] + df['ExitRates']) / 2

    # Is high value session?
    df['IsHighValue'] = (df['PageValues'] > df['PageValues'].median()).astype(int)

    # Month as ordered integer (Jan=1 .. Dec=12)
    month_order = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'June':6,
                   'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
    df['MonthNum'] = df['Month'].map(month_order).fillna(0).astype(int)

    # Q4 flag (high shopping season)
    df['IsQ4'] = df['MonthNum'].isin([10,11,12]).astype(int)

    print(f"[FEATURE ENG] {df.shape[1]} columns after engineering")
    return df

df_featured = task_feature_engineer(df_cleaned)
new_cols = ['TotalPages','TotalDuration','AvgDurationPerPage','ProductFocusRatio',
            'EngagementScore','IsHighValue','MonthNum','IsQ4']
df_featured[new_cols + ['Revenue']].describe().round(3)

[FEATURE ENG] 26 columns after engineering


,TotalPages,TotalDuration,AvgDurationPerPage,ProductFocusRatio,EngagementScore,IsHighValue,MonthNum,IsQ4,Revenue
count,12205.000,12205.000,12205.000,12205.000,12205.000,12205.000,12205.000,12205.000,12205.000
mean,34.893,1323.454,35.147,0.822,0.969,0.224,7.667,0.429,0.156
std,46.627,2043.872,33.838,0.155,0.045,0.417,3.387,0.495,0.363
min,0.000,0.000,0.000,0.000,0.800,0.000,2.000,0.000,0.000
25%,9.000,231.667,17.157,0.750,0.969,0.000,5.000,0.000,0.000
50%,20.000,690.958,28.150,0.875,0.984,0.000,8.000,0.000,0.000
75%,42.000,1643.958,43.400,0.934,0.992,0.000,11.000,1.000,0.000
max,746.000,69921.647,940.667,0.996,1.000,1.000,12.000,1.000,1.000


In [6]:
def task_encode(df: pd.DataFrame) -> pd.DataFrame:
    """Encode categorical variables."""
    df = df.copy()

    # Label encode VisitorType
    le = LabelEncoder()
    df['VisitorType_enc'] = le.fit_transform(df['VisitorType'])
    print(f"  VisitorType mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

    # One-hot encode Month (drop original)
    month_dummies = pd.get_dummies(df['Month'], prefix='Month', drop_first=True)
    df = pd.concat([df.drop(columns=['Month','VisitorType']), month_dummies], axis=1)

    # Convert bool columns to int
    bool_cols = df.select_dtypes(include='bool').columns
    df[bool_cols] = df[bool_cols].astype(int)

    print(f"[ENCODE] Final shape: {df.shape}")
    return df

df_encoded = task_encode(df_featured)
df_encoded.dtypes.value_counts()

  VisitorType mapping: {'New_Visitor': np.int64(0), 'Other': np.int64(1), 'Returning_Visitor': np.int64(2)}
[ENCODE] Final shape: (12205, 34)


int64      23
float64    11
Name: count, dtype: int64

In [7]:
def task_scale(df: pd.DataFrame) -> tuple:
    """Scale numerical features. Returns (scaled_df, scaler)."""
    target_col = 'Revenue'
    feature_cols = [c for c in df.columns if c != target_col]

    # Standard scaling on continuous features
    continuous = ['Administrative_Duration','Informational_Duration',
                  'ProductRelated_Duration','BounceRates','ExitRates',
                  'PageValues','TotalDuration','AvgDurationPerPage','EngagementScore']
    continuous = [c for c in continuous if c in df.columns]

    scaler = StandardScaler()
    df = df.copy()
    df[continuous] = scaler.fit_transform(df[continuous])

    print(f"[SCALE] Scaled {len(continuous)} continuous features")
    return df, scaler

df_scaled, scaler = task_scale(df_encoded)
df_scaled.head(2)

[SCALE] Scaled 9 continuous features


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,VisitorType_enc,Month_Dec,Month_Feb,Month_Jul,Month_June,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep
0,0,-0.460019,0,-0.246257,1,-0.628793,3.969402,3.434394,-0.318962,0.0,...,2,0,1,0,0,0,0,0,0,0
1,0,-0.460019,0,-0.246257,2,-0.595451,-0.450137,1.268054,-0.318962,0.0,...,2,0,1,0,0,0,0,0,0,0


In [8]:
def task_balance(df: pd.DataFrame) -> pd.DataFrame:
    """Apply SMOTE to handle class imbalance (Revenue: ~15% positive)."""
    X = df.drop(columns=['Revenue'])
    y = df['Revenue']

    print(f"[BALANCE] Before SMOTE — 0: {(y==0).sum()}  1: {(y==1).sum()}")
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X, y)
    print(f"[BALANCE] After  SMOTE — 0: {(y_res==0).sum()}  1: {(y_res==1).sum()}")

    df_balanced = pd.DataFrame(X_res, columns=X.columns)
    df_balanced['Revenue'] = y_res.values
    return df_balanced

df_balanced = task_balance(df_scaled)

[BALANCE] Before SMOTE — 0: 10297  1: 1908
[BALANCE] After  SMOTE — 0: 10297  1: 10297


## 3.3  Simulated Spark-Style Chunked Processing

In [9]:
# Mimic Spark partition processing using pandas chunks
CHUNK_SIZE = 2000
chunks = [df_balanced[i:i+CHUNK_SIZE] for i in range(0, len(df_balanced), CHUNK_SIZE)]

def process_partition(chunk: pd.DataFrame, partition_id: int) -> dict:
    """Compute aggregate stats per Spark partition."""
    return {
        'partition': partition_id,
        'rows': len(chunk),
        'revenue_rate': round(chunk['Revenue'].mean() * 100, 2),
        'avg_page_values': round(chunk['PageValues'].mean(), 4)
    }

partition_stats = [process_partition(c, i) for i, c in enumerate(chunks)]
pd.DataFrame(partition_stats)

,partition,rows,revenue_rate,avg_page_values
0,0,2000,9.30,-0.1208
1,1,2000,10.90,-0.0121
2,2,2000,13.25,-0.0200
3,3,2000,18.60,0.0711
4,4,2000,20.90,0.0563
5,5,2000,20.60,0.0240
6,6,2000,91.60,0.9760
7,7,2000,100.00,1.0946
8,8,2000,100.00,1.0846
9,9,2000,100.00,1.0811


## 3.4  LOAD — Write Processed Data

In [10]:
def task_load(df: pd.DataFrame):
    """Load processed data to SQL and Parquet."""
    # SQL
    conn = sqlite3.connect(DB_PATH)
    df.to_sql('processed_sessions', conn, if_exists='replace', index=False)
    conn.close()
    print(f"[LOAD] SQL — wrote {len(df):,} rows to 'processed_sessions'")

    # Parquet
    processed_dir = LAKE_DIR / 'processed'
    processed_dir.mkdir(exist_ok=True)
    df.to_parquet(processed_dir / 'processed_sessions.parquet', index=False)
    print(f"[LOAD] Parquet — saved to {processed_dir}")

task_load(df_balanced)

[LOAD] SQL — wrote 20,594 rows to 'processed_sessions'
[LOAD] Parquet — saved to C:\Stuffs\University Of New Haven\MSDS\Second Semester\Distributed and Scalable Data Engineering\Final\Project\Scripts\pipeline_project\pipeline\outputs\data_lake\processed


## 3.5  Simulated Airflow DAG

In [11]:
# This cell shows how the above tasks map to an Airflow DAG definition.
# In production you would put this in a dags/ folder and run via Airflow scheduler.
DAG_DEFINITION = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

default_args = {
    'owner': 'data_engineering',
    'retries': 2,
    'retry_delay': timedelta(minutes=5),
    'start_date': datetime(2024, 1, 1),
}

with DAG(
    dag_id='online_shopper_etl',
    schedule_interval='@daily',
    default_args=default_args,
    catchup=False,
) as dag:

    extract_task = PythonOperator(task_id='extract',         python_callable=task_extract)
    clean_task   = PythonOperator(task_id='clean',           python_callable=task_clean)
    feature_task = PythonOperator(task_id='feature_engineer',python_callable=task_feature_engineer)
    encode_task  = PythonOperator(task_id='encode',          python_callable=task_encode)
    scale_task   = PythonOperator(task_id='scale',           python_callable=task_scale)
    balance_task = PythonOperator(task_id='balance',         python_callable=task_balance)
    load_task    = PythonOperator(task_id='load',            python_callable=task_load)

    extract_task >> clean_task >> feature_task >> encode_task >> scale_task >> balance_task >> load_task
'''
print("Simulated Airflow DAG definition:")
print(DAG_DEFINITION)

Simulated Airflow DAG definition:

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

default_args = {
    'owner': 'data_engineering',
    'retries': 2,
    'retry_delay': timedelta(minutes=5),
    'start_date': datetime(2024, 1, 1),
}

with DAG(
    dag_id='online_shopper_etl',
    schedule_interval='@daily',
    default_args=default_args,
    catchup=False,
) as dag:

    extract_task = PythonOperator(task_id='extract',         python_callable=task_extract)
    clean_task   = PythonOperator(task_id='clean',           python_callable=task_clean)
    feature_task = PythonOperator(task_id='feature_engineer',python_callable=task_feature_engineer)
    encode_task  = PythonOperator(task_id='encode',          python_callable=task_encode)
    scale_task   = PythonOperator(task_id='scale',           python_callable=task_scale)
    balance_task = PythonOperator(task_id='balance',         python_callable=task_balance)
    load_

## Summary
| Step | Action | Output |
|---|---|---|
| Extract | SQL SELECT from raw_sessions | 12,330 rows |
| Clean | Drop metadata, fill nulls | |
| Feature Engineering | 8 new derived features | |
| Encode | LabelEncoder + OHE | |
| Scale | StandardScaler on continuous cols | |
| Balance | SMOTE oversampling | ~20,844 rows (balanced) |
| Load | SQL + Parquet | `processed_sessions` |